# Modeling Experiments

Ноутбук запускает воспроизводимый пайплайн из `src/models/train.py` и показывает, как сравниваются global popularity, segment popularity и supervised-модель. В текущей версии лучшей офлайн-моделью выбран segment popularity baseline.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.models.train import run_training
from src.utils.config import ProjectConfig

config = ProjectConfig()
config

ProjectConfig(project_root=PosixPath('/home/what/praktika/practicum-sem4-praktika1'), random_state=42, raw_data_path=PosixPath('/home/what/praktika/practicum-sem4-praktika1/train_ver2.csv'), processed_dir=PosixPath('/home/what/praktika/practicum-sem4-praktika1/data/processed'), eda_dir=PosixPath('/home/what/praktika/practicum-sem4-praktika1/data/eda_artifacts'), models_dir=PosixPath('/home/what/praktika/practicum-sem4-praktika1/models'), mlruns_dir=PosixPath('/home/what/praktika/practicum-sem4-praktika1/mlruns'), mlflow_tracking_uri='sqlite:///mlruns/mlflow.db', mlflow_experiment='bank-product-recommendations', train_months=('2015-02-28', '2015-03-28', '2015-04-28', '2015-05-28', '2015-06-28', '2015-07-28', '2015-08-28', '2015-09-28', '2015-10-28', '2015-11-28'), valid_months=('2015-12-28', '2016-01-28', '2016-02-28'), test_months=('2016-03-28', '2016-04-28'), chunk_size=200000, negative_sample_ratio=0.8, eval_month_sample_size=120000, top_k=3, monthly_dir_name='monthly', modeling_dir_

## 1. Запуск подготовки данных и обучения

In [2]:
bundle = run_training(config)
bundle['metrics']

KeyboardInterrupt: 

## 2. Смотрим артефакты обучения

In [ ]:
feature_importance = pd.read_csv(PROJECT_ROOT / 'models' / 'feature_importance.csv')
valid_errors = pd.read_csv(PROJECT_ROOT / 'models' / 'validation_errors.csv')
metadata = json.loads((PROJECT_ROOT / 'models' / 'model_metadata.json').read_text(encoding='utf-8'))

display(feature_importance.head(20))
display(valid_errors.head(10))
metadata

## 3. Интерпретация результатов

Что важно проверить после обучения:

1. В рекомендациях нет уже имеющихся продуктов: это обеспечивается маскированием текущего портфеля перед сортировкой.
2. Segment-based popularity baseline даёт лучший MAP@3/NDCG@3 на holdout-периодах и потому выбран для `best_model.joblib`.
3. Supervised SGD-модель сохранена как отдельный воспроизводимый эксперимент с анализом ошибок и коэффициентов.
4. Ошибки supervised-модели концентрируются на редких продуктах и нестандартных клиентских профилях.

## 4. Что можно улучшать дальше

- Больше лаговых признаков по продуктовой динамике.
- Отдельные product-specific модели или градиентный бустинг для редких продуктов.
- Калибровка score для онлайн-использования.
- Дообучение по скользящему окну с контролем drift.